# 07 · Comparar el troceo por estructura con nomic-embed-text en Qdrant

Medimos si trocear por la estructura Markdown (encabezados `#` a `####`) mejora la recuperación frente al tamaño fijo del notebook 06. Se fija el modelo (`nomic-embed-text`, Ollama), la base (Qdrant) y k=4. Cambian el troceo (estructura 1500/200 y 1000/200, con y sin la ruta de títulos en el texto embebido) y la referencia de tamaño fijo 1500 sin solape. Se evalúa con el golden set de `docs/GOLDEN_SET_PYTHON_DOCS.md` y las evidencias de `docs/golden_set_evidencias.json`.

Necesita Qdrant en `localhost:6333` (`docker compose up -d`) y Ollama con `nomic-embed-text` (`ollama pull nomic-embed-text`).

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

`CONFIGURACIONES` es el único sitio donde se declaran las configuraciones; el resto de celdas las recorre. En el troceo `fijo`, `minimo` es 0 y `con_ruta` es falso.

In [ ]:
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, evidencias_fuera_de_chunks, fuentes_ausentes,
    leer_evidencias, leer_golden_set, preparar_documento, preparar_query,
    rango_primer_acierto, rango_primer_acierto_evidencia, texto_a_embeber,
    trocear_corpus, trocear_por_estructura,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CSV = RAIZ / "data" / "resultados" / "troceo_estructura.csv"
RUTA_CSV_FICHA5 = RAIZ / "data" / "resultados" / "tamano_chunk.csv"
MODELO = "nomic"
K = 4
LOTE_EMBEDDING = 64   # chunks por llamada al modelo
LOTE_INSERCION = 500  # puntos por inserción
COLECCION_REFERENCIA = f"pydocs_{MODELO}_c1500_s0"
CONFIGURACIONES = [
    {"troceo": "estructura", "maximo": 1500, "minimo": 200, "con_ruta": True},
    {"troceo": "estructura", "maximo": 1500, "minimo": 200, "con_ruta": False},
    {"troceo": "estructura", "maximo": 1000, "minimo": 200, "con_ruta": True},
    {"troceo": "estructura", "maximo": 1000, "minimo": 200, "con_ruta": False},
    {"troceo": "fijo", "maximo": 1500, "minimo": 0, "con_ruta": False},
]

## 3. Conexión con Qdrant y Ollama

Se comprueba antes de calcular nada, para fallar pronto si falta un servicio. Si Qdrant u Ollama no responden, la celda para con un mensaje que dice qué hacer.

In [ ]:
qdrant = QdrantClient("localhost", port=6333)

try:

    qdrant.get_collections()

except Exception as error:

    raise RuntimeError(

        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d` y vuelve a ejecutar."

    ) from error



modelo_ollama = MODELOS[MODELO]["nombre"]

try:

    ollama.show(modelo_ollama)

except ConnectionError as error:

    raise RuntimeError("Ollama no responde. Arráncalo (`ollama serve` o la app) y vuelve a ejecutar.") from error

except ollama.ResponseError as error:

    raise RuntimeError(

        f"Ollama no tiene el modelo {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}` y vuelve a ejecutar."

    ) from error



print("Qdrant:", [col.name for col in qdrant.get_collections().collections])

print("Ollama:", modelo_ollama, "disponible")

## 4. Preguntas y evidencias del golden set

Se leen de las tablas de `docs/GOLDEN_SET_PYTHON_DOCS.md`; ninguna pregunta está escrita en el código. Antes de evaluar se avisa, con el ID y la ruta, de las fuentes que no existen en el corpus. La evidencia literal de cada pregunta A-D sale de `docs/golden_set_evidencias.json`.

In [ ]:
preguntas = leer_golden_set(RUTA_GOLDEN)

ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)

for id_, ruta in ausentes:

    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")

print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())



evidencias = leer_evidencias(RUTA_EVIDENCIAS)

## 5. Troceo por configuración

Se trocea el corpus una vez por cada `(troceo, maximo, minimo)` distinto: las variantes con y sin ruta comparten chunks. Se cuentan los chunks, el tamaño medio, los chunks de más de 6 000 caracteres y las evidencias que el troceo deja partidas. Por fichero, la concatenación de los chunks debe ser igual al fichero (RF-06); si no, la celda falla.

Los chunks indivisibles pueden superar el contexto de 2048 tokens de `nomic-embed-text`. No se recorta ningún texto en el notebook.

In [ ]:
def nombre_coleccion(c: dict) -> str:
    if c["troceo"] == "fijo":
        return f"pydocs_{MODELO}_c{c['maximo']}_s0"
    sufijo = "ruta" if c["con_ruta"] else "sinruta"
    return f"pydocs_{MODELO}_est{c['maximo']}_m{c['minimo']}_{sufijo}"


def clave_troceo(c: dict) -> tuple:
    return (c["troceo"], c["maximo"], c["minimo"])


def trocear(clave: tuple) -> list[dict]:
    troceo, maximo, minimo = clave
    if troceo == "fijo":
        return [{**c, "ruta_titulos": ""} for c in trocear_corpus(CARPETA_CORPUS, maximo, 0)]
    return trocear_por_estructura(CARPETA_CORPUS, maximo, minimo)


def comprobar_reproduce_ficheros(chunks: list[dict]) -> None:
    """Por fichero, la concatenación de los chunks es el fichero (solo con troceo sin solape)."""
    por_fuente: dict[str, list[str]] = {}
    for c in chunks:
        por_fuente.setdefault(c["fuente"], []).append(c["texto"])
    for fuente, textos in por_fuente.items():
        original = (CARPETA_CORPUS / fuente).read_text(encoding="utf-8")
        assert "".join(textos) == original, f"El troceo no reproduce {fuente}"


chunks_por_troceo: dict[tuple, list[dict]] = {}
partidas: dict[tuple, list[tuple[str, str]]] = {}
for config in CONFIGURACIONES:
    clave = clave_troceo(config)
    if clave in chunks_por_troceo:
        continue
    chunks = trocear(clave)
    chunks_por_troceo[clave] = chunks
    partidas[clave] = evidencias_fuera_de_chunks(evidencias, chunks)
    longitudes = pd.Series([len(c["texto"]) for c in chunks])
    print(f"{clave}: {len(chunks)} chunks, tamaño medio {longitudes.mean():.0f}, "
          f"{(longitudes > 6000).sum()} chunks de más de 6000 caracteres (máximo {longitudes.max()}), "
          f"{len(partidas[clave])} evidencias partidas")
    for id_, fuente in partidas[clave]:
        print(f"    partida: {id_} en {fuente}")
    if clave[0] == "estructura":
        comprobar_reproduce_ficheros(chunks)
        print("    reproduce los ficheros: sí")

## 6. Embeddings e indexación por configuración

Una colección por configuración, con distancia coseno. Las de estructura (`pydocs_nomic_est{maximo}_m{minimo}_ruta` y `_sinruta`) se borran y se recrean. La referencia `pydocs_nomic_c1500_s0` no se borra nunca (RF-10): si no existe, se crea; si existe, se reutiliza solo si tiene tantos puntos como chunks y `tamano_chunk.csv` tiene la fila 1500/0; si no, la celda para con un mensaje que dice qué falla. El texto embebido es `texto_a_embeber` más el prefijo de `MODELOS`; el payload guarda el texto original y la ruta de títulos.

Ollama trunca por defecto las entradas que pasan de su contexto (2048 tokens). Se comprobó con el chunk más largo (más de 200 000 caracteres): no da error y devuelve un vector. Esos chunks se embeben, por tanto, con solo su comienzo; el texto no se recorta aquí.

In [ ]:
def embeber(textos: list[str], barra: bool = False, descripcion: str = "") -> list[list[float]]:
    """Embeddings de una lista de textos ya preparados, en lotes fijos."""
    vectores: list[list[float]] = []
    for i in tqdm(range(0, len(textos), LOTE_EMBEDDING), desc=descripcion, disable=not barra):
        lote = textos[i:i + LOTE_EMBEDDING]
        vectores.extend(ollama.embed(model=modelo_ollama, input=lote).embeddings)
    return vectores


def indexar(nombre: str, chunks: list[dict], con_ruta: bool) -> float:
    inicio = time.perf_counter()
    vectores = embeber(
        [preparar_documento(MODELO, texto_a_embeber(c, con_ruta)) for c in chunks],
        barra=True, descripcion=nombre,
    )
    if qdrant.collection_exists(nombre):
        qdrant.delete_collection(nombre)
    qdrant.create_collection(
        nombre, vectors_config=VectorParams(size=len(vectores[0]), distance=Distance.COSINE)
    )
    for i in range(0, len(chunks), LOTE_INSERCION):
        qdrant.upsert(nombre, points=[
            PointStruct(id=c["id"], vector=v, payload={
                "fuente": c["fuente"], "texto": c["texto"], "ruta_titulos": c["ruta_titulos"],
            })
            for c, v in zip(chunks[i:i + LOTE_INSERCION], vectores[i:i + LOTE_INSERCION])
        ])
    return time.perf_counter() - inicio


def tiempo_referencia(chunks: list[dict]) -> float | None:
    """Tiempo de la referencia si hay que reutilizarla (None si hay que indexarla). Para si no cuadra."""
    nombre = COLECCION_REFERENCIA
    if not qdrant.collection_exists(nombre):
        return None
    puntos = qdrant.count(nombre, exact=True).count
    if puntos != len(chunks):
        raise RuntimeError(f"{nombre} existe pero tiene {puntos} puntos y el troceo da {len(chunks)} chunks. No la borro: revísala.")
    if not RUTA_CSV_FICHA5.exists():
        raise RuntimeError(f"{nombre} existe pero falta {RUTA_CSV_FICHA5.relative_to(RAIZ)}.")
    ficha5 = pd.read_csv(RUTA_CSV_FICHA5)
    fila = ficha5[(ficha5["tamano"] == 1500) & (ficha5["solape"] == 0)]
    if fila.empty:
        raise RuntimeError(f"{nombre} existe pero {RUTA_CSV_FICHA5.relative_to(RAIZ)} no tiene la fila 1500/0.")
    return float(fila.iloc[0]["indexación (s)"])


t_indexacion: dict[str, float] = {}
for config in CONFIGURACIONES:
    nombre = nombre_coleccion(config)
    chunks = chunks_por_troceo[clave_troceo(config)]
    if config["troceo"] == "fijo":
        reutilizado = tiempo_referencia(chunks)
        if reutilizado is not None:
            t_indexacion[nombre] = reutilizado
            print(f"{nombre}: referencia REUTILIZADA, indexación {reutilizado:.1f} s (de tamano_chunk.csv)")
            continue
        t_indexacion[nombre] = indexar(nombre, chunks, False)
        print(f"{nombre}: referencia INDEXADA, {len(chunks)} chunks en {t_indexacion[nombre]:.1f} s")
    else:
        t_indexacion[nombre] = indexar(nombre, chunks, config["con_ruta"])
        print(f"{nombre}: {len(chunks)} chunks en {t_indexacion[nombre]:.1f} s")

## 7. Evaluación con k=4

Para cada configuración y cada pregunta se recuperan 4 resultados y se mide la latencia (embedding de la query más búsqueda). Se calculan dos rangos: `rango` (acierta si la `fuente` del chunk es una de las fuentes de la pregunta) y `rango_evidencia` (además, el texto del chunk contiene la evidencia de esa fuente). Se guarda el score coseno del mejor resultado; en las preguntas E (sin fuentes) indica cuánta relevancia «inventa» el modelo.

In [ ]:
resultados = []
for config in CONFIGURACIONES:
    nombre = nombre_coleccion(config)
    for p in tqdm(preguntas, desc=nombre):
        inicio = time.perf_counter()
        vector = embeber([preparar_query(MODELO, p["query"])])[0]
        res = qdrant.query_points(nombre, query=vector, limit=K, with_payload=True).points
        latencia_ms = (time.perf_counter() - inicio) * 1000
        recuperados = [{"fuente": r.payload["fuente"], "texto": r.payload["texto"]} for r in res]
        resultados.append({
            "coleccion": nombre, "id": p["id"], "categoria": p["categoria"],
            "rango": rango_primer_acierto([r["fuente"] for r in recuperados], p["fuentes"]) if p["fuentes"] else None,
            "rango_evidencia": rango_primer_acierto_evidencia(recuperados, evidencias[p["id"]]) if p["fuentes"] else None,
            "con_fuentes": bool(p["fuentes"]),
            "score": res[0].score, "latencia_ms": latencia_ms,
        })

df = pd.DataFrame(resultados)
print(len(df), "evaluaciones")

## 8. Métricas por configuración (preguntas A-D)

hit@1, hit@4 y MRR@4 sobre las preguntas con fuentes, con los dos criterios: por fuente y por evidencia (más estricto, prefijo `ev_`).

In [ ]:
def metricas(grupo: pd.DataFrame) -> pd.Series:
    """Métricas por fuente y por evidencia (prefijo `ev_`)."""
    filas = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        filas.update({prefijo + n: v for n, v in calcular_metricas(rangos, K).items()})
    return pd.Series(filas)


con_fuentes = df[df["con_fuentes"]]
por_config = con_fuentes.groupby("coleccion").apply(metricas, include_groups=False)
por_config

## 9. Desglose por categoría (A-D)

In [ ]:
por_categoria = con_fuentes.groupby(["categoria", "coleccion"]).apply(metricas, include_groups=False)
por_categoria

## 10. Preguntas E: score del mejor resultado

Las preguntas E no tienen respuesta en el corpus. Un score alto en el mejor resultado indica que el modelo «inventa» relevancia.

In [ ]:
df[~df["con_fuentes"]].pivot_table(index="id", columns="coleccion", values="score").round(3)

## 11. Tabla resumen y CSV

Una fila por configuración. El tiempo de indexación es embeddings más inserción (en la referencia reutilizada, el de `tamano_chunk.csv`); el score E es la media del mejor score en las preguntas E. Se guarda en `data/resultados/troceo_estructura.csv` (la carpeta se crea si no existe).

In [ ]:
filas = []
for config in CONFIGURACIONES:
    nombre = nombre_coleccion(config)
    m = por_config.loc[nombre]
    sub = df[df["coleccion"] == nombre]
    filas.append({
        "troceo": config["troceo"], "maximo": config["maximo"], "minimo": config["minimo"],
        "con_ruta": config["con_ruta"], "coleccion": nombre,
        "chunks": len(chunks_por_troceo[clave_troceo(config)]),
        "tamaño medio (caracteres)": pd.Series([len(c["texto"]) for c in chunks_por_troceo[clave_troceo(config)]]).mean(),
        "hit@1": m["hit@1"], f"hit@{K}": m[f"hit@{K}"], "MRR": m["mrr"],
        "ev_hit@1": m["ev_hit@1"], f"ev_hit@{K}": m[f"ev_hit@{K}"], "ev_MRR": m["ev_mrr"],
        "latencia media (ms)": sub["latencia_ms"].mean(),
        "indexación (s)": t_indexacion[nombre],
        "score medio E": sub[~sub["con_fuentes"]]["score"].mean(),
        "evidencias partidas": len(partidas[clave_troceo(config)]),
    })
resumen = pd.DataFrame(filas)
assert not resumen.isna().any().any(), "El CSV tendría celdas vacías"
RUTA_CSV.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CSV, index=False)
print(f"Guardado en {RUTA_CSV.relative_to(RAIZ)}")
resumen.round(3)